In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
import pandas as pd
import uuid
import os

In [0]:

from datetime import datetime, timezone

dbutils.widgets.text(
    "source_path",
    "/Volumes/workspace/bronze/raw_files/world_bank_food_prices_2011_2026.csv"
)

dbutils.widgets.dropdown(
    "load_type",
    "incremental",
    ["full", "incremental", "backfill"]
)

dbutils.widgets.text("start_date", "2011-01-01")
dbutils.widgets.text("end_date", "2026-09-30")
dbutils.widgets.text("batch_id", "")

source_path = dbutils.widgets.get("source_path")
load_type = dbutils.widgets.get("load_type").lower()
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
batch_id = dbutils.widgets.get("batch_id")

run_id = str(uuid.uuid4())

if load_type not in ["full", "incremental", "backfill"]:
    raise ValueError("Invalid load_type")

datetime.strptime(start_date, "%Y-%m-%d")
datetime.strptime(end_date, "%Y-%m-%d")

if start_date > end_date:
    raise ValueError("start_date cannot exceed end_date")

if load_type == "full" and not batch_id:
    batch_id = "full_" + run_id

print("Source:", source_path)
print("Load type:", load_type)
print("Date range:", start_date, end_date)
print("Batch ID:", batch_id)
print("Run ID:", run_id)


In [0]:
import requests

url = "https://microdata.worldbank.org/api/tables/data/fcv/wld_2021_rtfp_v02_m"

response = requests.get(
    url,
    params={
        "limit": 10,
        "offset": 0,
        "format": "json"
    },
    timeout=120
)

print("Status:", response.status_code)
print(response.text[:1000])

In [0]:
import os
import csv
import json
import time
import requests

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------
url = "https://microdata.worldbank.org/api/tables/data/fcv/wld_2021_rtfp_v02_m"

base_path = "/Volumes/workspace/bronze/raw_files"
chunks_dir = f"{base_path}/download_chunks"
output_path = f"{base_path}/world_bank_food_prices_2011_2026.csv"

start_date = "2011-01-01"
end_date = "2026-09-30"
page_size = 1000

os.makedirs(chunks_dir, exist_ok=True)

session = requests.Session()

# --------------------------------------------------
# GET API RECORD COUNT
# --------------------------------------------------
response = session.get(
    url,
    params={"limit": 10, "offset": 0, "format": "json"},
    timeout=120
)
response.raise_for_status()

initial = response.json()
total_records = int(initial.get("total", initial.get("found")))

print("API records:", total_records)
print("Chunk directory:", chunks_dir)

# --------------------------------------------------
# DOWNLOAD AND FILTER EACH PAGE
# --------------------------------------------------
for offset in range(0, total_records, page_size):

    chunk_path = f"{chunks_dir}/page_{offset:09d}.json"

    # Skip a completed page.
    if os.path.exists(chunk_path):
        continue

    result = None

    for attempt in range(5):
        try:
            response = session.get(
                url,
                params={
                    "limit": page_size,
                    "offset": offset,
                    "format": "json"
                },
                timeout=180
            )
            response.raise_for_status()
            result = response.json()

            if not result.get("data"):
                raise ValueError(
                    f"Unexpected empty page at offset {offset}"
                )

            break

        except Exception:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)

    records = result["data"]

    filtered_records = [
        record for record in records
        if start_date <= str(record.get("price_date", "")) <= end_date
    ]

    # Write to a temporary page file first, then rename when complete.
    temp_path = chunk_path + ".tmp"

    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(filtered_records, file, ensure_ascii=False)

    os.replace(temp_path, chunk_path)

    if offset % 25000 == 0:
        print(
            "Offset:", offset,
            "| API rows:", len(records),
            "| Retained:", len(filtered_records)
        )

    time.sleep(0.1)

print("All pages downloaded.")

# --------------------------------------------------
# COMBINE PAGES INTO ONE CSV
# --------------------------------------------------
# Derive the full column list from a real API record.
columns = list(initial["data"][0].keys())

temp_output = output_path + ".tmp"
total_kept = 0

with open(temp_output, "w", newline="", encoding="utf-8") as output_file:
    writer = csv.DictWriter(
        output_file,
        fieldnames=columns,
        extrasaction="ignore"
    )
    writer.writeheader()

    for offset in range(0, total_records, page_size):
        chunk_path = f"{chunks_dir}/page_{offset:09d}.json"

        if not os.path.exists(chunk_path):
            raise FileNotFoundError(f"Missing chunk: {chunk_path}")

        with open(chunk_path, "r", encoding="utf-8") as file:
            records = json.load(file)

        writer.writerows(records)
        total_kept += len(records)

    output_file.flush()

os.replace(temp_output, output_path)

print("CSV saved:", output_path)
print("Retained rows:", total_kept)
print("CSV size (MB):", round(os.path.getsize(output_path) / (1024 * 1024), 2))

In [0]:
#for checking //validation
import os
import pandas as pd

source_path = (
    "/Volumes/workspace/bronze/raw_files/"
    "world_bank_food_prices_2011_2026.csv"
)

# 1. Check file
print("File exists:", os.path.exists(source_path))

if not os.path.exists(source_path):
    raise FileNotFoundError(source_path)

print(
    "File size (MB):",
    round(os.path.getsize(source_path) / (1024 * 1024), 2)
)

# 2. Check column count
header = pd.read_csv(source_path, nrows=0)
columns = header.columns.tolist()

print("Column count:", len(columns))
print("Required columns present:", all(
    c in columns for c in ["ISO3", "geo_id", "price_date"]
))

# 3. Validate dates and count rows in chunks
total_rows = 0
min_date = None
max_date = None
invalid_dates = 0
out_of_range = 0

for chunk in pd.read_csv(
    source_path,
    usecols=["price_date"],
    dtype=str,
    chunksize=50000
):
    dates = chunk["price_date"]
    total_rows += len(chunk)

    parsed = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
    invalid_dates += int(parsed.isna().sum())

    valid_dates = dates[parsed.notna()]

    if len(valid_dates) > 0:
        chunk_min = valid_dates.min()
        chunk_max = valid_dates.max()

        min_date = chunk_min if min_date is None else min(min_date, chunk_min)
        max_date = chunk_max if max_date is None else max(max_date, chunk_max)

    out_of_range += int(
        ((dates < "2011-01-01") | (dates > "2026-09-30")).sum()
    )

print("\n--- VALIDATION RESULTS ---")
print("Total rows:", total_rows)
print("Total columns:", len(columns))
print("Minimum date:", min_date)
print("Maximum date:", max_date)
print("Invalid dates:", invalid_dates)
print("Out-of-range dates:", out_of_range)
print("Expected row count:", 604602)
print("Row count matches:", total_rows == 604602)

In [0]:
#comparing bronze and source schemas
import pandas as pd

source_path = (
    "/Volumes/workspace/bronze/raw_files/"
    "world_bank_food_prices_2011_2026.csv"
)

# Read only the CSV header, not the full file
source_columns = pd.read_csv(source_path, nrows=0).columns.tolist()

# Read the existing Bronze table's column names
target_columns = spark.table(
    "bronze.world_bank_food_prices"
).columns

metadata_columns = [
    "_run_id",
    "_source_system",
    "_source_file",
    "_ingestion_timestamp",
    "load_timestamp"
]

target_source_columns = [
    column for column in target_columns
    if column not in metadata_columns
]

print("Source CSV columns:", len(source_columns))
print("Bronze table columns:", len(target_columns))
print("Bronze source columns:", len(target_source_columns))

print(
    "Missing in Bronze:",
    sorted(set(source_columns) - set(target_source_columns))
)

print(
    "Extra in Bronze:",
    sorted(set(target_source_columns) - set(source_columns))
)

print(
    "Column sets match:",
    set(source_columns) == set(target_source_columns)
)

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze.world_bank_food_prices")

# Show important column names
print("Key columns:")
for column in ["ISO3", "geo_id", "price_date"]:
    print(column, "->", bronze_df.schema[column].dataType)

print("\nColumns related to locations, prices, and commodities:")
for column in bronze_df.columns:
    name = column.lower()
    if any(word in name for word in [
        "country", "region", "market", "admin",
        "latitude", "longitude", "commodity",
        "price", "currency", "unit"
    ]):
        print(column, "->", bronze_df.schema[column].dataType)

# Display a small sample of records
sample_columns = [
    c for c in [
        "ISO3", "geo_id", "price_date",
        "country", "admin1", "market",
        "latitude", "longitude", "commodity"
    ]
    if c in bronze_df.columns
]

bronze_df.select(*sample_columns).show(5, truncate=False)

# Verify the business-key fields
bronze_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("ISO3").isNull(), 1).otherwise(0)).alias("null_ISO3"),
    F.sum(F.when(F.col("geo_id").isNull(), 1).otherwise(0)).alias("null_geo_id"),
    F.sum(F.when(F.col("price_date").isNull(), 1).otherwise(0)).alias("null_price_date")
).show()

In [0]:
data = response.json()

print(type(data))

if isinstance(data, dict):
    print(data.keys())

In [0]:
records = data["data"]

print("Number of records:", len(records))

display(records)

In [0]:
import pandas as pd

test_df = pd.DataFrame(records)

print(test_df.columns.tolist())

In [0]:
test_df["price_date"] = pd.to_datetime(
    test_df["price_date"],
    errors="coerce"
)

print(test_df["price_date"].min())
print(test_df["price_date"].max())

In [0]:
spark.sql("create schema if not exists bronze")
spark.sql("create schema if not exists silver")
spark.sql("create schema if not exists rejects")
spark.sql("create schema if not exists audit")

print("Schemas created.")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS audit.pipeline_execution_logs (
    run_id STRING,
    layer STRING,
    load_type STRING,
    parameter_processed STRING,
    source_file STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    rows_inserted BIGINT,
    rows_updated BIGINT,
    rows_processed BIGINT,
    error_message STRING
)
USING DELTA;

In [0]:
# =========================================================
# BRONZE AUDIT HELPER
# =========================================================

audit_table = "audit.pipeline_execution_logs"


def write_bronze_audit(
    run_id,
    load_type,
    source_path,
    start_time,
    end_time,
    status,
    rows_inserted,
    rows_updated,
    rows_processed,
    error_message
):
    audit_record = [(
        run_id,
        "Raw-to-Bronze",
        load_type,
        f"{start_date} to {end_date}",
        os.path.basename(source_path),
        start_time,
        end_time,
        status,
        rows_inserted,
        rows_updated,
        rows_processed,
        error_message
    )]

    audit_schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("layer", StringType(), False),
        StructField("load_type", StringType(), True),
        StructField("parameter_processed", StringType(), True),
        StructField("source_file", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("rows_processed", LongType(), True),
        StructField("error_message", StringType(), True)
    ])

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        audit_table
    )

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS rejects
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS rejects.bronze_schema_rejects (
    run_id STRING,
    source_file STRING,
    detected_at TIMESTAMP,
    reject_reason STRING,
    source_columns STRING,
    target_columns STRING
)
USING DELTA
""")

print("Bronze schema quarantine table is ready.")

In [0]:
spark.sql("show schemas").show()

In [0]:
# import requests
# import pandas as pd
# import time
# import os
# import json

# API_URL = "https://microdata.worldbank.org/api/tables/data/fcv/wld_2021_rtfp_v02_m"

# START_DATE = pd.Timestamp("2011-01-01")
# END_DATE = pd.Timestamp("2026-09-30")

# LIMIT = 1000

# OUTPUT_FILE = (
#     "/Volumes/workspace/bronze/raw_files/"
#     "world_bank_food_prices_2011_2026.csv"
# )

# os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
# # ---------------------------------------------------------
# # Delete old output if it exists
# # ---------------------------------------------------------


# # ---------------------------------------------------------
# # Variables
# # ---------------------------------------------------------

# offset = 0
# total_api_records = 0
# total_kept_records = 0
# total_pages = 0
# first_page = True

# # ---------------------------------------------------------
# # Download API data page by page
# # ---------------------------------------------------------

# while True:

#     print("\n========================================")
#     print("Requesting offset:", offset)
#     print("========================================")

#     # -----------------------------------------------------
#     # Request JSON because it contains the proper fields
#     # -----------------------------------------------------

#     try:

#         response = requests.get(
#             API_URL,
#             params={
#                 "limit": LIMIT,
#                 "offset": offset,
#                 "format": "json"
#             },
#             timeout=120
#         )

#         response.raise_for_status()

#     except requests.exceptions.RequestException as e:

#         print("Request failed:")
#         print(e)

#         print("Waiting 5 seconds before retrying...")
#         time.sleep(5)

#         continue

#     # -----------------------------------------------------
#     # Convert response to JSON
#     # -----------------------------------------------------

#     try:

#         data = response.json()

#     except Exception as e:

#         print("Could not read JSON response.")
#         print(e)

#         print("First 1000 characters of response:")
#         print(response.text[:1000])

#         break

#     # -----------------------------------------------------
#     # Find the actual records
#     # -----------------------------------------------------

#     if isinstance(data, list):

#         records = data

#     elif isinstance(data, dict):

#         if "data" in data:
#             records = data["data"]

#         elif "rows" in data:
#             records = data["rows"]

#         elif "results" in data:
#             records = data["results"]

#         else:

#             print("Could not identify data records.")
#             print("Available keys:")
#             print(data.keys())

#             break

#     else:

#         print("Unexpected API response format.")
#         print(type(data))

#         break

#     # -----------------------------------------------------
#     # Stop if no records
#     # -----------------------------------------------------

#     if len(records) == 0:

#         print("No more records.")
#         break

#     # -----------------------------------------------------
#     # Convert this API page to DataFrame
#     # -----------------------------------------------------

#     page_df = pd.DataFrame(records)

#     page_rows = len(page_df)

#     total_api_records += page_rows
#     total_pages += 1

#     print("Records received:", page_rows)

#     # -----------------------------------------------------
#     # Check price_date
#     # -----------------------------------------------------

#     if "price_date" not in page_df.columns:

#         print("\nERROR: price_date was not found.")

#         print("\nAvailable columns:")
#         print(page_df.columns.tolist())

#         print("\nFirst rows:")
#         display(page_df.head())

#         break

#     # -----------------------------------------------------
#     # Convert date
#     # -----------------------------------------------------

#     page_df["price_date"] = pd.to_datetime(
#         page_df["price_date"],
#         errors="coerce"
#     )

#     # -----------------------------------------------------
#     # Keep only latest 15 years
#     # -----------------------------------------------------

#     page_df = page_df[
#         (page_df["price_date"] >= START_DATE) &
#         (page_df["price_date"] <= END_DATE)
#     ].copy()

#     kept_rows = len(page_df)

#     total_kept_records += kept_rows

#     print("Records kept:", kept_rows)
#     print("Total records kept:", total_kept_records)

#     # -----------------------------------------------------
#     # Convert date back to CSV-friendly format
#     # -----------------------------------------------------

#     page_df["price_date"] = page_df["price_date"].dt.strftime(
#         "%Y-%m-%d"
#     )

#     # -----------------------------------------------------
#     # Write this page to CSV
#     # -----------------------------------------------------

#     if kept_rows > 0:

#         page_df.to_csv(
#             OUTPUT_FILE,
#             mode="w" if first_page else "a",
#             header=first_page,
#             index=False
#         )

#         first_page = False

#         print("Page written to CSV.")

#     # -----------------------------------------------------
#     # Check whether this is the last page
#     # -----------------------------------------------------

#     if page_rows < LIMIT:

#         print("Last API page reached.")
#         break

#     # -----------------------------------------------------
#     # Move to next page
#     # -----------------------------------------------------

#     offset += LIMIT

#     time.sleep(0.2)


# # =========================================================
# # FINAL SUMMARY
# # =========================================================

# print("\n")
# print("========================================")
# print("DOWNLOAD COMPLETED")
# print("========================================")

# print("Total API pages:", total_pages)
# print("Total API records:", total_api_records)
# print("Total records kept:", total_kept_records)

# print("CSV path:", OUTPUT_FILE)

# # ---------------------------------------------------------
# # Check output file
# # ---------------------------------------------------------

# if os.path.exists(OUTPUT_FILE):

#     file_size_mb = (
#         os.path.getsize(OUTPUT_FILE)
#         / (1024 * 1024)
#     )

#     print("CSV size:", round(file_size_mb, 2), "MB")

# else:

#     print("WARNING: CSV file was not created.")


# # =========================================================
# # VERIFY CSV
# # =========================================================

# if os.path.exists(OUTPUT_FILE):

#     print("\n")
#     print("========================================")
#     print("VERIFYING CSV")
#     print("========================================")

#     sample_df = pd.read_csv(
#         OUTPUT_FILE,
#         nrows=10
#     )

#     print("Number of columns:", len(sample_df.columns))

#     print("\nColumns:")
#     print(sample_df.columns.tolist())

#     print("\nSample rows:")

#     display(sample_df)


#     # -----------------------------------------------------
#     # Important columns
#     # -----------------------------------------------------

#     columns_to_show = [
#         "ISO3",
#         "country",
#         "adm1_name",
#         "adm2_name",
#         "mkt_name",
#         "geo_id",
#         "price_date",
#         "year",
#         "month",
#         "rice",
#         "wheat",
#         "maize",
#         "bread",
#         "milk",
#         "eggs"
#     ]

#     available_columns = [
#         column
#         for column in columns_to_show
#         if column in sample_df.columns
#     ]

#     print("\nImportant World Bank fields:")

#     display(
#         sample_df[available_columns]
#     )

In [0]:
spark.sql("create schema if not exists bronze")

spark.sql("""
create table if not exists bronze.test_storage (
    id int,
    message string
)
using delta
""")

spark.sql("""
insert into bronze.test_storage
values (1, 'persistent storage test')
""")

display(
    spark.sql("select * from bronze.test_storage")
)

In [0]:
# =========================================================
# IDEMPOTENCY TEST: BASELINE
# =========================================================

bronze_table = "bronze.world_bank_food_prices"

before_df = spark.table(bronze_table)

before_rows = before_df.count()

before_unique_keys = (
    before_df
    .select("ISO3", "geo_id", "price_date")
    .distinct()
    .count()
)

print("Rows before ingestion:", before_rows)
print("Unique keys before ingestion:", before_unique_keys)

In [0]:
import pandas as pd
import os
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)


# =========================================================
# SETTINGS
# =========================================================

file_path = source_path

CHUNK_SIZE = 5000

bronze_table = "bronze.world_bank_food_prices"
reject_table = "rejects.bronze_schema_rejects"

key_columns = ["ISO3", "geo_id", "price_date"]

metadata_columns = [
    "_run_id",
    "_source_system",
    "_source_file",
    "_ingestion_timestamp",
    "load_timestamp"
]


# =========================================================
# CHECK CSV
# =========================================================

print("Run ID:", run_id)
print("Load type:", load_type)

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"CSV file does not exist: {file_path}"
    )

print("CSV exists:", True)


# =========================================================
# READ CSV HEADER ONLY
# =========================================================

header_df = pd.read_csv(
    file_path,
    nrows=0
)

columns = header_df.columns.tolist()

print("Number of columns:", len(columns))


# =========================================================
# EXPLICIT SCHEMA
# =========================================================

schema = StructType([
    StructField(column, StringType(), True)
    for column in columns
])


# =========================================================
# CREATE REQUIRED SCHEMAS
# =========================================================

spark.sql("""
CREATE SCHEMA IF NOT EXISTS bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS rejects
""")


# =========================================================
# INITIALIZE PIPELINE
# =========================================================

table_exists = spark.catalog.tableExists(bronze_table)

total_rows = 0
chunk_number = 0

pipeline_start = datetime.now(timezone.utc).replace(
    tzinfo=None
)

print("Bronze table exists:", table_exists)


# =========================================================
# LOAD CSV IN CHUNKS
# =========================================================

for chunk in pd.read_csv(
    file_path,
    chunksize=CHUNK_SIZE,
    dtype=str,
    keep_default_na=False
):

    # -----------------------------------------------------
    # APPLY DATE FILTER FOR INCREMENTAL/BACKFILL
    # -----------------------------------------------------

    if load_type in ["incremental", "backfill"]:

        chunk_dates = pd.to_datetime(
            chunk["price_date"],
            errors="coerce"
        )

        chunk = chunk[
            (chunk_dates >= pd.Timestamp(start_date)) &
            (chunk_dates <= pd.Timestamp(end_date))
        ].copy()

    chunk_number += 1

    print("\nProcessing chunk:", chunk_number)
    print("Rows in chunk:", len(chunk))

    # Skip chunks that contain no rows after filtering
    if chunk.empty:
        print("No rows in this chunk after filtering.")
        continue

    # -----------------------------------------------------
    # MATCH COLUMN ORDER
    # -----------------------------------------------------

    chunk = chunk[columns]
    chunk = chunk.astype(str)

    # -----------------------------------------------------
    # CREATE SPARK DATAFRAME
    # -----------------------------------------------------

    spark_df = spark.createDataFrame(
        chunk,
        schema=schema
    )

    # -----------------------------------------------------
    # ADD BRONZE METADATA
    # -----------------------------------------------------

    spark_df = (
        spark_df
        .withColumn(
            "_run_id",
            F.lit(run_id)
        )
        .withColumn(
            "_source_system",
            F.lit("World Bank Real-Time Food Prices")
        )
        .withColumn(
            "_source_file",
            F.lit(os.path.basename(source_path))
        )
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "load_timestamp",
            F.current_timestamp()
        )
    )

    # -----------------------------------------------------
    # VALIDATE BUSINESS KEYS
    # -----------------------------------------------------

    invalid_key_condition = (
        F.col("ISO3").isNull() |
        (F.trim(F.col("ISO3")) == "") |
        F.col("geo_id").isNull() |
        (F.trim(F.col("geo_id")) == "") |
        F.col("price_date").isNull() |
        (F.trim(F.col("price_date")) == "")
    )

    invalid_key_count = (
        spark_df
        .filter(invalid_key_condition)
        .count()
    )

    if invalid_key_count > 0:
        raise ValueError(
            f"{invalid_key_count} records have invalid business keys."
        )

    # Check duplicate business keys within the current chunk
    duplicate_keys = (
        spark_df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicate_keys.limit(1).count() > 0:
        raise ValueError(
            "Duplicate business keys detected in the source chunk."
        )

    # =====================================================
    # FIRST CHUNK: CREATE BRONZE TABLE IF NEEDED
    # =====================================================

    if not table_exists:

        (
            spark_df.write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(bronze_table)
        )

        table_exists = True

        print("Bronze table created.")

    # =====================================================
    # EXISTING TABLE: VALIDATE SCHEMA AND MERGE
    # =====================================================

    else:

        # -------------------------------------------------
        # CHECK AND QUARANTINE SCHEMA DRIFT
        # -------------------------------------------------

        target_columns = spark.table(
            bronze_table
        ).columns

        if set(spark_df.columns) != set(target_columns):

            reject_record = [(
                run_id,
                os.path.basename(source_path),
                datetime.now(timezone.utc).replace(tzinfo=None),
                "Source and Bronze schemas differ",
                ",".join(spark_df.columns),
                ",".join(target_columns)
            )]

            reject_schema = StructType([
                StructField("run_id", StringType(), False),
                StructField("source_file", StringType(), True),
                StructField("detected_at", TimestampType(), True),
                StructField("reject_reason", StringType(), True),
                StructField("source_columns", StringType(), True),
                StructField("target_columns", StringType(), True)
            ])

            reject_df = spark.createDataFrame(
                reject_record,
                schema=reject_schema
            )

            (
                reject_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(reject_table)
            )

            raise ValueError(
                "Schema drift detected. Details saved to "
                f"{reject_table}."
            )

        # -------------------------------------------------
        # PREPARE DELTA MERGE
        # -------------------------------------------------

        bronze_delta = DeltaTable.forName(
            spark,
            bronze_table
        )

        # Business columns exclude keys and metadata
        business_columns = [
            column
            for column in columns
            if column not in key_columns
        ]

        # Update only when business data has changed
        changed_condition = " OR ".join([
            (
                f"NOT (target.`{column.replace('`', '``')}` "
                f"<=> source.`{column.replace('`', '``')}`)"
            )
            for column in business_columns
        ])

        # Update business columns but do not modify keys
        update_map = {
            column: (
                f"source.`{column.replace('`', '``')}`"
            )
            for column in business_columns
        }

        # Update metadata when business data changes
        for column in metadata_columns:
            update_map[column] = f"source.`{column}`"

        # Insert all columns for new records
        insert_map = {
            column: (
                f"source.`{column.replace('`', '``')}`"
            )
            for column in spark_df.columns
        }

        # -------------------------------------------------
        # EXECUTE DELTA MERGE
        # -------------------------------------------------

        (
            bronze_delta.alias("target")
            .merge(
                spark_df.alias("source"),
                """
                target.ISO3 = source.ISO3
                AND target.geo_id = source.geo_id
                AND target.price_date = source.price_date
                """
            )
            .whenMatchedUpdate(
                condition=changed_condition,
                set=update_map
            )
            .whenNotMatchedInsert(
                values=insert_map
            )
            .execute()
        )

        print("Conditional MERGE completed.")
        print("Chunk merged successfully.")

    # -----------------------------------------------------
    # UPDATE PROCESSING COUNT
    # -----------------------------------------------------

    total_rows += len(chunk)

    print("Chunk loaded successfully.")
    print("Total source rows processed:", total_rows)


# =========================================================
# COMPLETED
# =========================================================

print("\n========================================")
print("BRONZE LOAD COMPLETED")
print("========================================")

print("Total source rows processed:", total_rows)
print("Run ID:", run_id)


# =========================================================
# WRITE SUCCESS AUDIT
# =========================================================

pipeline_end = datetime.now(timezone.utc).replace(
    tzinfo=None
)

write_bronze_audit(
    run_id=run_id,
    load_type=load_type,
    source_path=source_path,
    start_time=pipeline_start,
    end_time=pipeline_end,
    status="SUCCESS",
    rows_inserted=None,
    rows_updated=None,
    rows_processed=total_rows,
    error_message=None
)

print("Bronze success audit recorded.")

In [0]:
# =========================================================
# IDEMPOTENCY TEST: AFTER RE-RUN
# =========================================================

after_df = spark.table(bronze_table)

after_rows = after_df.count()

after_unique_keys = (
    after_df
    .select("ISO3", "geo_id", "price_date")
    .distinct()
    .count()
)

print("Rows before:", before_rows)
print("Rows after:", after_rows)

print("Unique keys before:", before_unique_keys)
print("Unique keys after:", after_unique_keys)

if after_rows != before_rows:
    raise ValueError(
        "Idempotency failed: total row count changed."
    )

if after_unique_keys != after_rows:
    raise ValueError(
        "Duplicate business keys detected."
    )

print("Idempotency row-count check passed.")

In [0]:
%sql
select
    ISO3,
    country,
    geo_id,
    price_date,
    rice,
    wheat,
    _run_id,
    _source_system,
    _ingestion_timestamp
from bronze.world_bank_food_prices
limit 10;

In [0]:
%sql
select
    count(*) as total_rows,
    count_if(rice is not null and trim(rice) <> '') as rice_values,
    count_if(wheat is not null and trim(wheat) <> '') as wheat_values,
    count_if(maize is not null and trim(maize) <> '') as maize_values
from bronze.world_bank_food_prices;

In [0]:
%sql
select
    count(*) as total_rows,
    count_if(rice is not null and trim(rice) <> '') as rice_values,
    count_if(wheat is not null and trim(wheat) <> '') as wheat_values,
    count_if(maize is not null and trim(maize) <> '') as maize_values,
    count_if(bread is not null and trim(bread) <> '') as bread_values,
    count_if(milk is not null and trim(milk) <> '') as milk_values,
    count_if(eggs is not null and trim(eggs) <> '') as eggs_values
from bronze.world_bank_food_prices;

In [0]:
%sql
select
    count(*) as total_rows,

    count_if(trim(rice) <> '') as rice_non_empty,
    count_if(trim(wheat) <> '') as wheat_non_empty,
    count_if(trim(maize) <> '') as maize_non_empty,
    count_if(trim(bread) <> '') as bread_non_empty,
    count_if(trim(milk) <> '') as milk_non_empty,
    count_if(trim(eggs) <> '') as eggs_non_empty,

    count_if(trim(inflation_food_price_index) <> '') as food_index_non_empty

from bronze.world_bank_food_prices;

In [0]:
%sql
select
    count(*) as total_rows,
    count(distinct concat(ISO3, '|', geo_id, '|', price_date)) as unique_keys
from bronze.world_bank_food_prices;

World Bank Bronze status

Your Bronze layer now has:

604,602 total records
604,602 unique business keys
0 duplicate business keys
Business key:
ISO3 + geo_id + price_date

In [0]:
# spark.sql("""
# alter table bronze.world_bank_food_prices
# add columns (load_timestamp timestamp)
# """)

In [0]:
spark.sql("""
update bronze.world_bank_food_prices
set load_timestamp = _ingestion_timestamp
where load_timestamp is null
""")

In [0]:
display(
    spark.sql("""
        select
            count(*) as total_rows,
            count(load_timestamp) as rows_with_load_timestamp
        from bronze.world_bank_food_prices
    """)
)

In [0]:
#verification of load_timestamp
display(
    spark.sql("""
        select
            ISO3,
            country,
            geo_id,
            price_date,
            load_timestamp,
            _run_id,
            _source_file
        from bronze.world_bank_food_prices
        limit 10
    """)
)

In [0]:
print("Bronze parameter test")
print("source_path =", source_path)
print("load_type =", load_type)
print("run_id =", run_id)

In [0]:
from delta.tables import DeltaTable

bronze_table = "bronze.world_bank_food_prices"

bronze_delta = DeltaTable.forName(
    spark,
    bronze_table
)

# Take 5 complete existing Bronze records
merge_test_df = (
    spark.table(bronze_table)
    .limit(5)
)

(
    bronze_delta.alias("target")
    .merge(
        merge_test_df.alias("source"),
        """
        target.ISO3 = source.ISO3
        AND target.geo_id = source.geo_id
        AND target.price_date = source.price_date
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE test completed successfully.")

In [0]:
display(
    spark.sql("""
        describe bronze.world_bank_food_prices
    """)
)